# OAuth 인증을 사용하는 AWS Agent Registry

이 Notebook에서는 AWS Agent Registry에 OAuth 인증을 구성하고 사용하는 방법을 보여 줍니다. OAuth는 AWS Agent Registry 검색 작업에 안전한 토큰 기반 인증을 제공합니다.

![아키텍처 흐름](images/registry-end-to-end-oauth.png)

## 학습 내용

- AWS Cognito를 OAuth 공급자로 설정(user pool, app client, 테스트 사용자)
- `CUSTOM_JWT` 권한 부여(discovery URL 및 허용된 클라이언트)를 사용하는 **Agent Registry** 생성
- **레지스트리 레코드** 생성 및 관리(MCP 설명자 유형)
- 레코드 상태 전환 관리(DRAFT → PENDING_APPROVAL → APPROVED)
- Cognito에서 JWT 액세스 토큰 발급
- Bearer 토큰을 사용하여 승인된 레코드의 **시맨틱 검색** 수행
- 인증되지 않은 요청이 거부되는지 확인

## 인증 유형

AWS Agent Registry는 두 가지 인바운드 인증 유형을 지원합니다.

| 유형 | 설명 |
|:-----|:------------|
| `AWS_IAM` | 기본값. 모든 검색 요청에 IAM SigV4 서명을 사용합니다. |
| `CUSTOM_JWT` | JWT Bearer Token 인증을 사용합니다. OpenID Connect discovery URL과 허용된 클라이언트 ID 목록이 필요합니다. |

하나의 Agent Registry에는 IAM 또는 JWT 인증 중 하나만 사용할 수 있으며 두 인증을 동시에 사용할 수 없습니다. 인증 유형이 서로 다른 별도의 레지스트리를 생성할 수 있습니다.

이 Notebook에서는 AWS Cognito를 자격 증명 공급자로 사용하는 `CUSTOM_JWT` 흐름을 보여 줍니다.

## 아키텍처 흐름

```
Cognito                  Agent Registry                    소비자
───────                  ──────────────                    ───────
  │                           │                               │
  │  1. User pool 생성        │                               │
  │     + app client          │                               │
  │     + 테스트 사용자        │                               │
  │                           │                               │
  │                    2. Registry 생성                       │
  │                       (CUSTOM_JWT 인증                    │
  │                        + discovery URL                    │
  │                        + 허용된 클라이언트)                  │
  │                           │                               │
  │                    3. MCP 레코드 생성                     │
  │                       + 승인                              │
  │                           │                               │
  │  4. 사용자 인증           │                               │
  │<──────────────────────────────────────────────────────────│
  │                           │                               │
  │  5. JWT 토큰 반환         │                               │
  │──────────────────────────────────────────────────────────>│
  │                           │                               │
  │                           │  6. Bearer 토큰으로 검색      │
  │                           │<──────────────────────────────│
  │                           │                               │
  │                           │  7. 토큰 검증 + APPROVED      │
  │                           │     레코드 반환               │
  │                           │──────────────────────────────>│
  │                           │                               │
```

## 설정

### 사전 요구 사항

- IAM 자격 증명이 구성된 AWS 계정
- Python 3.10+
- `boto3 >= 1.42.87`
- 아래 권한이 있는 IAM 사용자 또는 역할(`ACCOUNT_ID`와 `REGION`을 필요에 맞게 변경)


<details>
<summary>필수 IAM 정책(클릭하여 펼치기)</summary>

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowCreateRegistry",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:CreateRegistry"],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:*"]
        },
        {
            "Sid": "AllowGetUpdateDeleteRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistry",
                "bedrock-agentcore:DeleteRegistry"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowCreateAndListRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistryRecord",
                "bedrock-agentcore:SearchRegistryRecords"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowRecordOperations",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:DeleteRegistryRecord",
                "bedrock-agentcore:SubmitRegistryRecordForApproval"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*/record/*"]
        }
    ]
}
```

</details>

**참고:** 이 Notebook은 OAuth 설정의 일부로 Amazon Cognito 리소스(User Pool, App Client, 테스트 사용자)를 생성합니다. 기존 Cognito 구성은 필요하지 않습니다.

### 종속성 설치

In [ ]:
!pip install -r requirements.txt

### AWS Session 및 클라이언트 초기화

레지스트리 관리 및 검색 작업을 위한 boto3 클라이언트를 구성합니다.

In [ ]:
import boto3
import json
import time
from boto3.session import Session

# 구성
boto_session = Session()
AWS_REGION = boto_session.region_name

# AWS_PROFILE = "aws-profile"  # 사용할 프로파일로 변경합니다. SageMaker에서 실행하는 경우 이 줄을 주석 처리하세요

# Amazon SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명 설정
# os.environ["AWS_PROFILE"] = AWS_PROFILE

# boto3 Session 생성
# boto_session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)  # SageMaker에서 실행하지 않는 경우 사용

# 레지스트리 관리용 클라이언트
registry_client = boto_session.client("bedrock-agentcore-control", region_name=AWS_REGION)

# Registry 검색 엔드포인트
REGISTRY_SEARCH_ENDPOINT = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/registry-records/search"

print(f"Session ready | Region: {AWS_REGION}")

### 헬퍼 함수

Agent Registry 생성은 비동기 작업입니다. 이 헬퍼는 Agent Registry가 `READY` 상태가 될 때까지 폴링합니다.

In [ ]:
# 터미널 출력용 ANSI 색상
class C:
    GREEN = "\033[92m"
    RED = "\033[91m"
    YELLOW = "\033[93m"
    CYAN = "\033[96m"
    BOLD = "\033[1m"
    DIM = "\033[2m"
    RESET = "\033[0m"


def wait_for_registry(registry_id, interval=5):
    while True:
        resp = registry_client.get_registry(registryId=registry_id)
        status = resp["status"]
        if status == "READY":
            print(f"  {C.GREEN}✅ Registry Status: {status}{C.RESET}")
            resp.pop("ResponseMetadata", None)
            print(json.dumps(resp, indent=2, default=str))
            return resp
        if status.endswith("_FAILED"):
            print(f"  {C.RED}❌ Registry Status: {status}{C.RESET}")
            raise Exception(f"Registry failed: {status} - {resp.get('statusReason')}")
        print(f"  {C.YELLOW}⏳ Registry Status: {status}{C.RESET}")
        time.sleep(interval)


def pretty_print_response(response):
    """API 응답에서 ResponseMetadata를 제외하고 보기 좋게 출력합니다."""
    data = {k: v for k, v in response.items() if k != "ResponseMetadata"}
    print(json.dumps(data, indent=2, default=str))

---
## 1. OAuth 공급자 구성(AWS Cognito)

Agent Registry에서 JWT 인증을 사용하려면 OpenID Connect(OIDC) 자격 증명 공급자가 필요합니다. 여기서는 다음 기능을 제공하는 AWS Cognito를 사용합니다.
- **User Pool** — JWT 토큰을 발급하는 자격 증명 저장소
- **App Client** — 허용된 인증 흐름을 정의하는 애플리케이션 등록
- **Discovery URL** — Agent Registry가 토큰을 검증하는 데 사용하는 OIDC 엔드포인트

### 1.1 Cognito User Pool 생성 또는 가져오기

Cognito User Pool을 생성하거나 기존 Pool을 재사용합니다. 들어오는 JWT 토큰을 검증할 수 있도록 Pool의 OIDC discovery URL을 Agent Registry에 구성합니다.

In [ ]:
USER_POOL_NAME = "agentcore-registry-pool"

cognito = boto3.client("cognito-idp", region_name=AWS_REGION)

# 기존 Pool 찾기
pools = cognito.list_user_pools(MaxResults=60)["UserPools"]
existing_pool = next((p for p in pools if p["Name"] == USER_POOL_NAME), None)

if existing_pool:
    user_pool_id = existing_pool["Id"]
    print(f"  {C.YELLOW}⚠️  Using existing pool: {user_pool_id}{C.RESET}")
else:
    # 새 Pool 생성
    user_pool = cognito.create_user_pool(PoolName=USER_POOL_NAME)["UserPool"]
    user_pool_id = user_pool["Id"]

    cognito.create_user_pool_domain(Domain=user_pool_id.replace("_", "").lower(), UserPoolId=user_pool_id)
    print(f"  {C.GREEN}✅ User pool created{C.RESET}")

discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

print(f"  {C.BOLD}Pool ID:{C.RESET}       {C.CYAN}{user_pool_id}{C.RESET}")
print(f"  {C.BOLD}Discovery URL:{C.RESET}  {C.CYAN}{discovery_url}{C.RESET}")

### 1.2 App Client 생성

`USER_PASSWORD_AUTH` 흐름이 활성화된 Cognito App Client를 생성합니다. Client ID는 Agent Registry의 `allowedClients` 목록에 추가됩니다. 허용된 클라이언트에 발급된 토큰만 레지스트리에 액세스할 수 있습니다.

In [ ]:
CLIENT_NAME = "agentcore-registry-client"

# App Client 생성 또는 가져오기
clients = cognito.list_user_pool_clients(UserPoolId=user_pool_id)["UserPoolClients"]
existing_client = next((c for c in clients if c["ClientName"] == CLIENT_NAME), None)

if existing_client:
    client_id = existing_client["ClientId"]
    print(f"  {C.YELLOW}⚠️  Using existing client: {client_id}{C.RESET}")
else:
    client = cognito.create_user_pool_client(
        UserPoolId=user_pool_id,
        ClientName=CLIENT_NAME,
        GenerateSecret=False,
        ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
    )
    client_id = client["UserPoolClient"]["ClientId"]
    print(f"  {C.GREEN}✅ App client created{C.RESET}")

print(f"  {C.BOLD}Client ID:{C.RESET}  {C.CYAN}{client_id}{C.RESET}")

### 1.3 테스트 사용자 생성

Cognito User Pool에 테스트 사용자를 생성합니다. 이 사용자는 Agent Registry를 검색하기 위한 JWT 토큰을 인증하고 발급받습니다.

In [ ]:
TEST_USERNAME = "testuser"
TEST_PASSWORD = "TempPass123!"

# 테스트 사용자 생성
try:
    cognito.admin_create_user(
        UserPoolId=user_pool_id,
        Username=TEST_USERNAME,
        TemporaryPassword=TEST_PASSWORD,
        MessageAction="SUPPRESS",
    )
    cognito.admin_set_user_password(
        UserPoolId=user_pool_id,
        Username=TEST_USERNAME,
        Password=TEST_PASSWORD,
        Permanent=True,
    )
    print(f"  {C.GREEN}✅ Test user created: {TEST_USERNAME}{C.RESET}")
except cognito.exceptions.UsernameExistsException:
    print(f"  {C.YELLOW}⚠️  User {TEST_USERNAME} already exists{C.RESET}")

---
## 2. OAuth 구성을 사용하는 Agent Registry 생성

`CUSTOM_JWT` authorizer 유형으로 Agent Registry를 생성합니다. 이 구성을 통해 Agent Registry는 JWT 토큰을 사용하여 들어오는 검색 요청을 검증합니다.

`authorizerConfiguration`에는 다음 항목이 필요합니다.
- **`discoveryUrl`** — OIDC discovery 엔드포인트(예: Cognito의 `.well-known/openid-configuration` URL). Agent Registry는 이를 사용하여 토큰 서명 검증을 위한 JWKS(JSON Web Key Set)를 가져옵니다.
- **`allowedClients`** — 토큰이 허용되는 app client ID 목록. 목록에 없는 클라이언트의 토큰은 거부됩니다.

In [ ]:
create_registry_respone = registry_client.create_registry(
    name="RegistryWithOauth",
    description="Registry created from Jupyter notebook with OAuth",
    approvalConfiguration={"autoApproval": False},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
        }
    },
)

REGISTRY_ARN = create_registry_respone["registryArn"]
REGISTRY_ID = REGISTRY_ARN.split("/")[-1]

wait_for_registry(REGISTRY_ID)

print(f"  {C.GREEN}✅ Registry created!{C.RESET}")
print(f"  {C.BOLD}ARN:{C.RESET}        {C.CYAN}{REGISTRY_ARN}{C.RESET}")
print(f"  {C.BOLD}ID:{C.RESET}         {C.CYAN}{REGISTRY_ID}{C.RESET}")
print(f"  {C.BOLD}Auth Type:{C.RESET}   {C.CYAN}CUSTOM_JWT{C.RESET}")

---
## 3. 레지스트리 레코드 생성 및 승인

### 3.1 MCP 레코드 생성

샘플 MCP 서버 레코드를 Agent Registry에 등록합니다. MCP 레코드에는 다음 항목이 포함됩니다.
- **`server`** — 서버 메타데이터(이름, 설명, 버전, 패키지, 전송 구성)
- **`tools`** — 서버의 기능을 설명하는 입력 스키마가 포함된 도구 정의

레코드는 승인 워크플로를 거친 후 검색할 수 있습니다.

In [ ]:
mcp_server_schema = json.dumps(
    {
        "name": "io.example/weather-server",
        "description": "A weather data MCP server that provides current conditions and forecasts",
        "version": "1.0.0",
        "title": "Weather Server",
        "websiteUrl": "https://example.com/weather",
        "packages": [
            {
                "registryType": "npm",
                "identifier": "@example/weather-mcp",
                "version": "1.0.0",
                "registryBaseUrl": "https://registry.npmjs.org",
                "runtimeHint": "npx",
                "transport": {"type": "stdio"},
                "environmentVariables": [
                    {
                        "name": "WEATHER_API_KEY",
                        "description": "API key for the weather service",
                        "isSecret": True,
                    }
                ],
            }
        ],
    }
)

mcp_tool_schema = json.dumps(
    {
        "tools": [
            {
                "name": "get_current_weather",
                "description": "Get current weather for a city",
                "inputSchema": {
                    "type": "object",
                    "properties": {"city": {"type": "string", "description": "City name"}},
                    "required": ["city"],
                },
            },
            {
                "name": "get_forecast",
                "description": "Get 5-day weather forecast",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name"},
                        "days": {
                            "type": "integer",
                            "description": "Number of forecast days",
                        },
                    },
                    "required": ["city"],
                },
            },
        ]
    }
)

mcp_record_response = registry_client.create_registry_record(
    registryId=REGISTRY_ID,
    name="weather_server",
    description="MCP server providing weather data and forecasts",
    descriptorType="MCP",
    descriptors={
        "mcp": {
            "server": {
                "schemaVersion": "2025-12-11",
                "inlineContent": mcp_server_schema,
            },
            "tools": {
                "protocolVersion": "2025-11-25",
                "inlineContent": mcp_tool_schema,
            },
        }
    },
    recordVersion="1.0",
)

MCP_RECORD_ARN = mcp_record_response["recordArn"]
MCP_RECORD_ID = MCP_RECORD_ARN.split("/")[-1]
print(f"  {C.GREEN}✅ MCP Record created: {C.CYAN}{MCP_RECORD_ID}{C.RESET}")

### 3.2 레코드 나열

Agent Registry에 레코드가 생성되었는지 확인합니다. 처음에는 `DRAFT` 상태이므로 아직 검색할 수 없습니다.

In [ ]:
records_response = registry_client.list_registry_records(registryId=REGISTRY_ID)

print(f"{C.BOLD}=== Registry Records ==={C.RESET}")
print(f"Found {len(records_response['registryRecords'])} record(s):\n")
for rec in records_response["registryRecords"]:
    status = rec["status"]
    sc = C.GREEN if status == "APPROVED" else C.YELLOW if status in ("DRAFT", "PENDING_APPROVAL") else C.RED
    print(
        f"  {sc}[{status}]{C.RESET} {rec['name']} | {C.CYAN}{rec['descriptorType']}{C.RESET} | {C.DIM}{rec['recordId']}{C.RESET}"
    )

### 3.3 레코드 승인

레코드의 승인을 요청한 다음 승인합니다. `APPROVED` 레코드만 검색할 수 있습니다.

```
DRAFT → PENDING_APPROVAL → APPROVED(이제 검색 가능)
                         → REJECTED
                         → DEPRECATED(승인 후 사용 중단된 경우)
```

프로덕션 환경에서는 게시자가 레코드를 제출하고 별도의 관리자가 검토하고 승인합니다. 여기서는 두 단계를 모두 수행합니다.

In [ ]:
# 1단계: 게시자가 승인 요청
registry_client.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=MCP_RECORD_ID)
print(f"  {C.YELLOW}⏳ MCP record → PENDING_APPROVAL{C.RESET}")

# 2단계: 관리자가 승인
registry_client.update_registry_record_status(
    registryId=REGISTRY_ID,
    recordId=MCP_RECORD_ID,
    statusReason="Approved by admin",
    status="APPROVED",
)
print(f"  {C.GREEN}✅ MCP record → APPROVED{C.RESET}")

### 3.4 레코드 상태 확인

레코드가 이제 `APPROVED` 상태이며 Agent Registry를 통해 인증된 검색을 수행할 준비가 되었는지 확인합니다.

In [ ]:
record_response = registry_client.get_registry_record(registryId=REGISTRY_ID, recordId=MCP_RECORD_ID)
status = record_response["status"]
sc = C.GREEN if status == "APPROVED" else C.YELLOW if status in ("DRAFT", "PENDING_APPROVAL") else C.RED
print(f"{C.BOLD}=== Record Details ==={C.RESET}")
print(f"  {C.BOLD}Name:{C.RESET}      {C.CYAN}{record_response['name']}{C.RESET}")
print(f"  {C.BOLD}Protocol:{C.RESET}   {C.CYAN}{record_response['descriptorType']}{C.RESET}")
print(f"  {C.BOLD}Status:{C.RESET}     {sc}{status}{C.RESET}")
print(f"  {C.BOLD}Version:{C.RESET}    {C.CYAN}{record_response['recordVersion']}{C.RESET}")
print(f"\n{C.BOLD}Full descriptor:{C.RESET}")
pretty_print_response(record_response)

---
## 4. 인증 및 액세스 토큰 발급

`USER_PASSWORD_AUTH` 흐름을 사용하여 Cognito에서 테스트 사용자를 인증하고 JWT 액세스 토큰을 발급받습니다.

Agent Registry에 대한 모든 검색 요청에서 이 토큰을 `Bearer {token}` 형식으로 `Authorization` 헤더에 포함해야 합니다.

In [ ]:
# 사용자를 인증하고 액세스 토큰 발급
try:
    cognito = boto3.client("cognito-idp", region_name=AWS_REGION)
    auth_response = cognito.initiate_auth(
        ClientId=client_id,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={"USERNAME": TEST_USERNAME, "PASSWORD": TEST_PASSWORD},
    )

    bearer_token = auth_response["AuthenticationResult"]["AccessToken"]

    print(f"  {C.GREEN}✅ Authentication successful!{C.RESET}")
    print(f"  {C.BOLD}Access Token:{C.RESET}  {C.DIM}{bearer_token[:50]}...{C.RESET}")
    print(f"  {C.BOLD}Token Type:{C.RESET}    {C.CYAN}Bearer{C.RESET}")

except cognito.exceptions.NotAuthorizedException:
    print(f"  {C.RED}❌ Authentication failed: Invalid username or password{C.RESET}")
except cognito.exceptions.UserNotConfirmedException:
    print(f"  {C.RED}❌ User not confirmed. Please confirm user first.{C.RESET}")
except Exception as e:
    print(f"  {C.RED}❌ Authentication error: {e}{C.RESET}")

---
## 5. 인증된 검색 수행

JWT 액세스 토큰을 사용하여 Agent Registry를 검색합니다. Agent Registry는 결과를 반환하기 전에 토큰을 검증합니다.

1. Discovery URL에서 가져온 JWKS를 사용하여 토큰 서명 검증
2. 토큰 만료 여부 확인
3. `client_id` 클레임이 `allowedClients` 목록에 있는지 검증
4. 검색 쿼리와 일치하는 `APPROVED` 레코드만 반환

유효한 토큰이 없으면 검색 요청이 `401 Unauthorized` 오류로 거부됩니다.

In [ ]:
import requests
import json


def search_registry_records(access_token, search_query, registry_identifiers, max_results=10):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}",
    }

    payload = {
        "searchQuery": search_query,
        "registryIds": registry_identifiers,
        "maxResults": max_results,
    }

    response = requests.post(f"{REGISTRY_SEARCH_ENDPOINT}", headers=headers, json=payload, timeout=30)
    return response.json()


# 검색 인덱스가 업데이트될 때까지 대기
print(f"  {C.YELLOW}⏳ Waiting 100s for search index...{C.RESET}")
time.sleep(100)

# 사용 예제
access_token = bearer_token  # Cognito 인증에서 발급받은 값
registry_arn = REGISTRY_ARN

results = search_registry_records(access_token, "weather", [registry_arn], 10)

print(f"{C.BOLD}🔍 Search: 'weather'{C.RESET}")
records = results.get("registryRecords", [])
if records:
    print(f"  {C.GREEN}✅ Found {len(records)} result(s){C.RESET}")
else:
    print(f"  {C.YELLOW}⚠️  No results found{C.RESET}")
print(json.dumps(results, indent=2))

### 5.1 유효한 토큰 없이 검색(부정 테스트)

유효한 Bearer 토큰 없이 검색을 시도하여 Agent Registry가 인증을 적용하는지 확인합니다. 두 요청 모두 오류를 반환해야 합니다.

In [ ]:
import json

# 토큰 없이 검색 시도
print(f"{C.BOLD}=== Negative Auth Tests ==={C.RESET}")
print(f"\n{C.BOLD}Test 1: Search without Authorization header{C.RESET}")
headers_no_auth = {"Content-Type": "application/json"}
payload = {"searchQuery": "weather", "registryIds": [REGISTRY_ARN], "maxResults": 10}

response = requests.post(f"{REGISTRY_SEARCH_ENDPOINT}", headers=headers_no_auth, json=payload, timeout=30)
sc = C.GREEN if response.status_code in (401, 403) else C.RED
print(f"  {sc}Status Code: {response.status_code}{C.RESET}")
print(f"  {C.DIM}{response.text}{C.RESET}\n")

# 유효하지 않은 토큰으로 검색 시도
print(f"{C.BOLD}Test 2: Search with invalid token{C.RESET}")
headers_invalid = {
    "Content-Type": "application/json",
    "Authorization": "Bearer invalid-token-12345",
}

response = requests.post(f"{REGISTRY_SEARCH_ENDPOINT}", headers=headers_invalid, json=payload, timeout=30)
sc = C.GREEN if response.status_code in (401, 403) else C.RED
print(f"  {sc}Status Code: {response.status_code}{C.RESET}")
print(f"  {C.DIM}{response.text}{C.RESET}\n")

print(f"  {C.GREEN}✅ Both requests correctly rejected{C.RESET}")

---
## 6. 정리(선택 사항)

### 6.1 Agent Registry 및 레코드 삭제

모든 레코드와 Agent Registry를 삭제하여 리소스를 정리합니다. 이 단계는 선택 사항입니다. 추가 실험을 위해 Agent Registry를 유지할 수도 있습니다.

In [ ]:
# 레지스트리의 모든 레코드 삭제
records = registry_client.list_registry_records(registryId=REGISTRY_ID)
for rec in records.get("registryRecords", []):
    record_id = rec["recordId"]
    registry_client.delete_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    print(f"  {C.GREEN}✅ Deleted record: {C.DIM}{record_id}{C.RESET}")

# 레지스트리 삭제
registry_client.delete_registry(registryId=REGISTRY_ID)
print(f"  {C.GREEN}✅ Deleted registry: {C.DIM}{REGISTRY_ID}{C.RESET}")

print(f"\n{C.GREEN}✅ Registry cleanup complete!{C.RESET}")

### 6.2 Cognito 리소스 삭제

Cognito User Pool, App Client, 테스트 사용자를 삭제합니다. 이 단계는 선택 사항입니다. 여러 Agent Registry에서 OAuth를 테스트할 때 이러한 리소스를 재사용할 수 있습니다.

In [ ]:
# 테스트 사용자 삭제
try:
    cognito.admin_delete_user(UserPoolId=user_pool_id, Username=TEST_USERNAME)
    print(f"  {C.GREEN}✅ Deleted user: {TEST_USERNAME}{C.RESET}")
except Exception as e:
    print(f"  {C.RED}❌ Error deleting user: {e}{C.RESET}")

# App Client 삭제
try:
    cognito.delete_user_pool_client(UserPoolId=user_pool_id, ClientId=client_id)
    print(f"  {C.GREEN}✅ Deleted app client: {C.DIM}{client_id}{C.RESET}")
except Exception as e:
    print(f"  {C.RED}❌ Error deleting app client: {e}{C.RESET}")

# User Pool을 삭제하기 전에 필요한 User Pool 도메인 삭제
try:
    domain = user_pool_id.replace("_", "").lower()
    cognito.delete_user_pool_domain(Domain=domain, UserPoolId=user_pool_id)
    print(f"  {C.GREEN}✅ Deleted user pool domain: {C.DIM}{domain}{C.RESET}")
except Exception as e:
    print(f"  {C.RED}❌ Error deleting domain: {e}{C.RESET}")

# User Pool 삭제
try:
    cognito.delete_user_pool(UserPoolId=user_pool_id)
    print(f"  {C.GREEN}✅ Deleted user pool: {C.DIM}{user_pool_id}{C.RESET}")
except Exception as e:
    print(f"  {C.RED}❌ Error deleting user pool: {e}{C.RESET}")

print(f"\n{C.GREEN}✅ Cognito cleanup complete!{C.RESET}")